#### 1. The first 17 steps are from the GDP_workers.ipynb file. 
#### 2. The below cells are processed using the output generated from the step. 
#### 3. This processing follows the GDP per capita values for every NUTS 3 region. 

In [ ]:
import pandas as pd

# Step 1: Read the combined pollutant and GDP data (with geometry) file
df_combined = pd.read_csv("combined_pollutant_gdp_data_with_geometry.csv")

# Step 2: Read the population density data (ensure it's properly formatted)
df_population = pd.read_csv("population.csv")

# Step 3: Reshape the population data to have 'NUTS_ID', 'year', and 'population' columns
df_population = df_population.melt(id_vars=["NUTS_ID"], 
                                   value_vars=["2018", "2019", "2020", "2021", "2022", "2023"],
                                   var_name="year", value_name="population")

# Ensure 'year' column is integer type for proper merging
df_population['year'] = df_population['year'].astype(int)

# Step 4: Merge population data with the combined pollutant and GDP data
df_combined = df_combined.merge(df_population, on=['NUTS_ID', 'year'], how='left')

# Step 5: Calculate GDP per capita
df_combined['GDP'] = df_combined['GDP'] *1000000

df_combined['GDP_per_capita'] = df_combined['GDP'] / df_combined['population']

# Step 6: Check for any NaN values in GDP_per_capita column (optional)
print("Missing values in GDP_per_capita:", df_combined['GDP_per_capita'].isna().sum())

# Step 7: Save the updated DataFrame with GDP_per_capita
df_combined.to_csv("combined_pollutant_gdp_data_with_gdp_per_capita1.csv", index=False)

print("GDP per capita column added and saved as 'combined_pollutant_gdp_data_with_gdp_per_capita.csv'")


In [ ]:
df_combined

In [ ]:
df_final_filled = df_combined

In [ ]:
import pandas as pd

# Define the pollution concentration ranges and their corresponding numerical labels
def categorize_pollutant(value, pollutant_type):
    # Categorize O₃ (mol/m³)
    if pollutant_type == 'O3':
        if value <= 0.05:
            return 1  # Very Good
        elif value <= 0.10:
            return 2  # Good
        elif value <= 0.15:
            return 3  # Medium
        elif value <= 0.20:
            return 4  # Poor
        elif value <= 0.30:
            return 5  # Very Poor
        else:
            return 6  # Extremely Poor
    
    # Categorize CO (mol/m³)
    elif pollutant_type == 'CO':
        if value <= 0.01:
            return 1  # Very Good
        elif value <= 0.02:
            return 2  # Good
        elif value <= 0.03:
            return 3  # Medium
        elif value <= 0.05:
            return 4  # Poor
        elif value <= 0.10:
            return 5  # Very Poor
        else:
            return 6  # Extremely Poor
    
    # Categorize NO₂ (µg/m³)
    elif pollutant_type == 'NO2':
        if value <= 50:
            return 1  # Very Good
        elif value <= 100:
            return 2  # Good
        elif value <= 200:
            return 3  # Medium
        elif value <= 500:
            return 4  # Poor
        elif value <= 1000:
            return 5  # Very Poor
        else:
            return 6  # Extremely Poor
    
    # Categorize SO₂ (µg/m³)
    elif pollutant_type == 'SO2':
        if value <= 50:
            return 1  # Very Good
        elif value <= 100:
            return 2  # Good
        elif value <= 150:
            return 3  # Medium
        elif value <= 200:
            return 4  # Poor
        elif value <= 300:
            return 5  # Very Poor
        else:
            return 6  # Extremely Poor
    
    # Categorize PM2.5 (µg/m³)
    elif pollutant_type == 'PM25':
        if value <= 10:
            return 1  # Very Good
        elif value <= 15:
            return 2  # Good
        elif value <= 20:
            return 3  # Medium
        elif value <= 30:
            return 4  # Poor
        elif value <= 50:
            return 5  # Very Poor
        else:
            return 6  # Extremely Poor
    
    # Categorize HCHO (µg/m³)
    elif pollutant_type == 'HCHO':
        if value <= 5:
            return 1  # Very Good
        elif value <= 10:
            return 2  # Good
        elif value <= 15:
            return 3  # Medium
        elif value <= 20:
            return 4  # Poor
        elif value <= 30:
            return 5  # Very Poor
        else:
            return 6  # Extremely Poor

# Assuming df_final_filled is the dataframe with the pollutant data

# Apply the categorization to each pollutant
df_final_filled['O3_quality'] = df_final_filled['O3'].apply(categorize_pollutant, pollutant_type='O3')
df_final_filled['CO_quality'] = df_final_filled['CO'].apply(categorize_pollutant, pollutant_type='CO')
df_final_filled['NO2_quality'] = df_final_filled['NO2'].apply(categorize_pollutant, pollutant_type='NO2')
df_final_filled['SO2_quality'] = df_final_filled['SO2'].apply(categorize_pollutant, pollutant_type='SO2')
df_final_filled['PM25_quality'] = df_final_filled['PM25'].apply(categorize_pollutant, pollutant_type='PM25')
df_final_filled['HCHO_quality'] = df_final_filled['HCHO'].apply(categorize_pollutant, pollutant_type='HCHO')

# Now, the dataframe includes the numerical quality categories for each pollutant
# You can optionally save this to a new CSV file
df_final_filled.to_csv("combined_pollutant_quality_data_numerical1.csv", index=False)

print("Combined dataframe with numerical pollutant quality labels saved as 'combined_pollutant_quality_data_numerical.csv'")


In [ ]:
import pandas as pd

# Define seasonal multipliers for each pollutant
seasonal_multipliers = {
    'Winter': {'PM25': 0.40, 'NO2': 0.25, 'O3': 0.10, 'SO2': 0.12, 'CO': 0.06, 'HCHO': 0.07},
    'Spring': {'PM25': 0.36, 'NO2': 0.22, 'O3': 0.15, 'SO2': 0.12, 'CO': 0.07, 'HCHO': 0.08},
    'Summer': {'PM25': 0.25, 'NO2': 0.15, 'O3': 0.30, 'SO2': 0.05, 'CO': 0.10, 'HCHO': 0.15},
    'Autumn': {'PM25': 0.35, 'NO2': 0.23, 'O3': 0.15, 'SO2': 0.12, 'CO': 0.07, 'HCHO': 0.08}
}

# Function to map months to seasons
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

# Apply the function to create a season column
df_final_filled['season'] = df_final_filled['month'].apply(get_season)

# Create a new set of weighted quality columns for each pollutant
df_final_filled['PM25_weighted_quality'] = df_final_filled.apply(lambda row: row['PM25_quality'] * seasonal_multipliers[row['season']]['PM25'], axis=1)
df_final_filled['NO2_weighted_quality'] = df_final_filled.apply(lambda row: row['NO2_quality'] * seasonal_multipliers[row['season']]['NO2'], axis=1)
df_final_filled['O3_weighted_quality'] = df_final_filled.apply(lambda row: row['O3_quality'] * seasonal_multipliers[row['season']]['O3'], axis=1)
df_final_filled['SO2_weighted_quality'] = df_final_filled.apply(lambda row: row['SO2_quality'] * seasonal_multipliers[row['season']]['SO2'], axis=1)
df_final_filled['CO_weighted_quality'] = df_final_filled.apply(lambda row: row['CO_quality'] * seasonal_multipliers[row['season']]['CO'], axis=1)
df_final_filled['HCHO_weighted_quality'] = df_final_filled.apply(lambda row: row['HCHO_quality'] * seasonal_multipliers[row['season']]['HCHO'], axis=1)

# Now, the dataframe includes the weighted quality columns
# You can save this updated dataframe to a new CSV file if needed
df_final_filled.to_csv("combined_pollutant_weighted_quality_data.csv", index=False)

print("Combined dataframe with weighted quality columns saved as 'combined_pollutant_weighted_quality_data1.csv'")


In [ ]:
# Create a new column 'Air_quality_Index' which is the sum of the weighted quality columns
df_final_filled['Index'] = df_final_filled['PM25_weighted_quality'] + \
                                       df_final_filled['NO2_weighted_quality'] + \
                                       df_final_filled['O3_weighted_quality'] + \
                                       df_final_filled['SO2_weighted_quality'] + \
                                       df_final_filled['CO_weighted_quality'] + \
                                       df_final_filled['HCHO_weighted_quality']

# Print the dataframe to verify the new column
print(df_final_filled[['PM25_weighted_quality', 'NO2_weighted_quality', 'O3_weighted_quality', 
                       'SO2_weighted_quality', 'CO_weighted_quality', 'HCHO_weighted_quality', 
                       'Index']].head())


In [ ]:
# Step 1: Create a helper column to extract the country code from the NUTS_ID
# Assuming NUTS_ID is a string where the first two characters represent the country
df_final_filled['Country'] = df_final_filled['NUTS_ID'].str[:2]

# Step 2: Define a function to calculate the normalized GDP for each NUTS3 region
def normalize_gdp(group):
    # Calculate GDPMin and GDPMax for each country and year group
    GDPMin = group['GDP'].min()
    GDPMax = group['GDP'].max()
    
    # Apply the formula to calculate GDP_Normalized
    group['GDP_Normalized'] = 1 - ((group['GDP'] - GDPMin) / (GDPMax - GDPMin))
    
    return group

# Step 3: Group the dataframe by 'Country' and 'year' and apply the normalization function
df_final_filled = df_final_filled.groupby(['Country', 'year']).apply(normalize_gdp)

# Step 4: Print a few rows to verify the new column 'GDP_Normalized'
print(df_final_filled[['NUTS_ID', 'year', 'GDP', 'GDP_Normalized']].head())


In [ ]:
# Step 1: Extract the country from NUTS_ID (first two letters)
df_final_filled['Country'] = df_final_filled['NUTS_ID'].str[:2]

# Reset the index without adding index columns back
df_final_filled = df_final_filled.reset_index(drop=True)

# Step 2: Calculate GDP min and max for each country and year
df_final_filled['GDP_per_capita_min'] = df_final_filled.groupby(['Country', 'year'])['GDP_per_capita'].transform('min')
df_final_filled['GDP_per_capita_max'] = df_final_filled.groupby(['Country', 'year'])['GDP_per_capita'].transform('max')

# Check the result
print(df_final_filled[['NUTS_ID', 'year', 'Country', 'GDP_per_capita', 'GDP_per_capita_min', 'GDP_per_capita_max']].head())


In [ ]:
df_final_filled.columns

In [ ]:
# Step 1: Create a helper column to extract the country code from the NUTS_ID
# Assuming NUTS_ID is a string where the first two characters represent the country
df_final_filled['Country'] = df_final_filled['NUTS_ID'].str[:2]

# Step 2: Define a function to calculate the normalized GDP for each NUTS3 region
def normalize_gdp(group):
    # Calculate GDPMin and GDPMax for each country and year group
    GDPMin = group['GDP_per_capita'].min()
    GDPMax = group['GDP_per_capita'].max()
    
    # Apply the formula to calculate GDP_Normalized
    group['GDP_Normalized'] = 1 - ((group['GDP_per_capita'] - GDPMin) / (GDPMax - GDPMin))
    
    return group

# Step 3: Group the dataframe by 'Country' and 'year' and apply the normalization function
df_final_filled = df_final_filled.groupby(['Country', 'year']).apply(normalize_gdp)

# Step 4: Print a few rows to verify the new column 'GDP_Normalized'
df_final_filled

In [ ]:
df_final_filled.columns

In [ ]:
df_final_filled.to_csv('final_v2.csv')

In [ ]:
gdf = df_final_filled

In [ ]:
gdf['geometry'] = df['geometry'].apply(wkt.loads)
# Convert the DataFrame into a GeoDataFrame, specifying the geometry column and a CRS (e.g., EPSG:4326)
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

In [ ]:
gd = gpd.GeoDataFrame(gdf, geometry='geometry_gdf', crs="EPSG:4326")

In [ ]:
gd = gd.drop(["GDP"], axis=1)

In [ ]:
gd.to_file("final_v2.shp", driver="ESRI Shapefile")

In [ ]:
gdf.columns

In [ ]:
import geopandas as gpd

# Read the shapefile
gdf = gpd.read_file("final_v2.shp")

# Convert and save to GeoJSON
gdf.to_file("out.geojson", driver="GeoJSON")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
file_path = "final_v2.csv"
df = pd.read_csv(file_path)

# 1. Scatter Plot: GDP per Capita vs Air Inequity Index
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="GDP_per_capita", y="Air_Inequity_Index", alpha=0.6)
plt.xlabel("GDP per Capita")
plt.ylabel("Air Inequity Index")
plt.title("Relationship Between GDP per Capita and Air Inequity")
plt.savefig("gdp_vs_inequity.png")
plt.show()

# 2. Temporal Trend of Air Inequity Index
plt.figure(figsize=(10, 5))
sns.lineplot(data=df.groupby("year")["Air_Inequity_Index"].mean().reset_index(), x="year", y="Air_Inequity_Index", marker="o")
plt.xlabel("Year")
plt.ylabel("Average Air Inequity Index")
plt.title("Temporal Trends in Air Inequity Index")
plt.savefig("temporal_inequity.png")
plt.show()

# 3. Heatmap: Air Quality vs GDP per Capita
plt.figure(figsize=(8, 6))
sns.heatmap(df[["GDP_per_capita", "PM25", "NO2", "SO2", "CO", "O3", "Air_Inequity_Index"]].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Between GDP, API, and AII")
plt.savefig("correlation_heatmap.png")
plt.show()

# 4. Box Plot: Seasonal Variation in Air Inequity Index
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x="season", y="Air_Inequity_Index")
plt.xlabel("Season")
plt.ylabel("Air Inequity Index")
plt.title("Seasonal Variation in Air Inequity Index")
plt.savefig("seasonal_variation.png")
plt.show()

# 5. Bar Chart: Country-wise Average Air Inequity Index
plt.figure(figsize=(12, 6))
df_grouped = df.groupby("Country")["Air_Inequity_Index"].mean().sort_values()
df_grouped.plot(kind="bar", color="teal")
plt.xlabel("Country")
plt.ylabel("Average Air Inequity Index")
plt.title("Country-wise Air Inequity Index")
plt.xticks(rotation=45)
plt.savefig("country_inequity.png")
plt.show()

# 6. Pair Plot: Air Pollutants vs Air Inequity Index
sns.pairplot(df, vars=["PM25", "NO2", "SO2", "CO", "O3", "Air_Inequity_Index"], diag_kind="kde")
plt.suptitle("Pairwise Relationships Between Air Pollutants and Air Inequity Index", y=1.02)
plt.savefig("pairplot_air_pollutants.png")
plt.show()

In [ ]:
# 7. Violin Plot: Distribution of Air Inequity Index by Country
plt.figure(figsize=(12, 6))
sns.violinplot(data=df, x="Country", y="Air_Inequity_Index")
plt.xlabel("Country")
plt.ylabel("Air Inequity Index")
plt.title("Distribution of Air Inequity Index by Country")
plt.xticks(rotation=45)
plt.savefig("violin_air_inequity.png")
plt.show()

# 8. Histogram: Distribution of Air Inequity Index
plt.figure(figsize=(8, 6))
sns.histplot(df["Air_Inequity_Index"], bins=30, kde=True, color="blue")
plt.xlabel("Air Inequity Index")
plt.ylabel("Frequency")
plt.title("Distribution of Air Inequity Index")
plt.savefig("histogram_air_inequity.png")
plt.show()

# 9. Regression Plot: GDP per Capita vs Air Inequity Index
plt.figure(figsize=(8, 6))
sns.regplot(data=df, x="GDP_per_capita", y="Air_Inequity_Index", scatter_kws={"alpha":0.5})
plt.xlabel("GDP per Capita")
plt.ylabel("Air Inequity Index")
plt.title("Regression Analysis: GDP per Capita vs Air Inequity Index")
plt.savefig("regression_gdp_inequity.png")
plt.show()


In [ ]:


# 10. Bar Chart: Top 10 Cities with Highest Air Inequity Index
plt.figure(figsize=(12, 6))
df_sorted = df.sort_values("Air_Inequity_Index", ascending=False).head(10)
sns.barplot(data=df_sorted, x="NUTS_ID", y="Air_Inequity_Index", palette="Reds")
plt.xlabel("City")
plt.ylabel("Air Inequity Index")
plt.title("Top 10 Cities with Highest Air Inequity Index")
plt.xticks(rotation=45)
plt.savefig("top10_cities_inequity.png")
plt.show()

# 11. KDE Plot: Air Inequity Index Distribution by GDP Category
plt.figure(figsize=(8, 6))
sns.kdeplot(data=df, x="Air_Inequity_Index", hue=pd.qcut(df["GDP_per_capita"], q=4, labels=["Low", "Medium", "High", "Very High"]), fill=True)
plt.xlabel("Air Inequity Index")
plt.ylabel("Density")
plt.title("Density Plot of Air Inequity Index by GDP Category")
plt.savefig("kde_gdp_inequity.png")
plt.show()

# 12. Swarm Plot: Air Inequity Index by Continent
plt.figure(figsize=(12, 6))
sns.swarmplot(data=df, x="Country", y="Air_Inequity_Index", palette="coolwarm")
plt.xlabel("Continent")
plt.ylabel("Air Inequity Index")
plt.title("Air Inequity Index Distribution by Countries")
plt.savefig("swarm_continent_inequity.png")
plt.show()





In [ ]:
plt.figure(figsize=(8, 6))
plt.hexbin(df["GDP_per_capita"], df["Air_Inequity_Index"], gridsize=30, cmap="Blues", mincnt=1)
plt.colorbar(label="Count")
plt.xlabel("GDP per Capita")
plt.ylabel("Air Inequity Index")
plt.title("Hexbin Plot: GDP per Capita vs Air Inequity Index")
plt.savefig("hexbin_gdp_inequity.png")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
file_path = "final_v2.csv"
df = pd.read_csv(file_path)

# 1. Binned Line Plot: GDP per Capita vs Air Inequity Index
bins = pd.qcut(df["GDP_per_capita"], q=20, duplicates='drop')
grouped = df.groupby(bins)["Air_Inequity_Index"].mean().reset_index()

plt.figure(figsize=(8, 6))
plt.plot(grouped["GDP_per_capita"].apply(lambda x: x.mid), grouped["Air_Inequity_Index"], marker='o', linestyle='-', color='blue')
plt.xlabel("GDP per Capita (Binned)")
plt.ylabel("Average Air Inequity Index")
plt.title("Binned Line Plot: GDP per Capita vs Air Inequity Index")
plt.grid(True)
plt.savefig("binned_line_gdp_inequity.png")
plt.show()


In [ ]:
# 9. Regression Plot: GDP per Capita vs Air Inequity Index
plt.figure(figsize=(8, 6))
sns.regplot(data=df, x="GDP_per_capita", y="Index", scatter_kws={"alpha":0.5})
plt.xlabel("GDP per Capita")
plt.ylabel("Air Inequity Index")
plt.title("Regression Analysis: GDP per Capita vs Air Inequity Index")
plt.savefig("regression_gdp_inequity.png")
plt.show()

In [ ]:
# 2. Box Plot: Diversity of Air Inequity Index Across NUTS_ID
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="NUTS_ID", y="Air_Inequity_Index")
plt.xlabel("NUTS ID")
plt.ylabel("Air Inequity Index")
plt.title("Diversity of Air Inequity Index Across NUTS ID")
plt.xticks(rotation=90)
plt.savefig("nuts_air_inequity_diversity.png")
plt.show()

In [ ]:
# 2. Box Plot: Diversity of Air Inequity Index Across NUTS_ID with Different Colors for Countries
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="NUTS_ID", y="Air_Inequity_Index", hue="Country", palette="tab10")
plt.xlabel("NUTS ID")
plt.ylabel("Air Inequity Index")
plt.title("Diversity of Air Inequity Index Across NUTS ID by Country")
plt.xticks(rotation=90)
plt.legend(title="Country", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig("nuts_air_inequity_diversity_colored.png", bbox_inches='tight')
plt.show()


In [ ]:
df_sorted = df.groupby("NUTS_ID")["Air_Inequity_Index"].median().sort_values().index
df["NUTS_ID"] = pd.Categorical(df["NUTS_ID"], categories=df_sorted, ordered=True)

plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="NUTS_ID", y="Air_Inequity_Index", hue="Country", palette="tab10")
plt.xlabel("NUTS ID")
plt.ylabel("Air Inequity Index")
plt.title("Diversity of Air Inequity Index Across NUTS ID by Country (Sorted)")
plt.xticks(rotation=90)
plt.legend(title="Country", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig("nuts_air_inequity_diversity_sorted.png", bbox_inches='tight')
plt.show()


In [ ]:
# 2. Box Plot: Diversity of Air Inequity Index Across NUTS_ID with Different Colors for Countries
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="NUTS_ID", y="GDP_per_capita", hue="Country", palette="tab10")
plt.xlabel("NUTS ID")
plt.ylabel("GDP_per_capita")
plt.title("Diversity of GDP_per_capita Across NUTS ID by Country")
plt.xticks(rotation=90)
plt.legend(title="Country", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig("nuts_GDP_per_capita_diversity_colored.png", bbox_inches='tight')
plt.show()


In [ ]:
# 2. Box Plot: Diversity of Air Inequity Index Across NUTS_ID with Different Colors for Countries
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="NUTS_ID", y="Air_Inequity_Index", hue="Country", palette="tab10")
plt.xlabel("NUTS ID")
plt.ylabel("Air Inequity Index")
plt.title("Diversity of Air Inequity Index Across NUTS ID by Country")
plt.xticks(rotation=90)
plt.legend(title="Country", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig("nuts_air_inequity_diversity_colored.png", bbox_inches='tight')
plt.show()


In [ ]:
df

In [ ]:
df